# Notebook 22 — Set-level interaction and recovery-reordering benchmark

The G5 failure leaves an unresolved causal question:

> Does channel-level validity fail because pruning harm is non-additive at the set
> level, or because post-pruning recovery reorders otherwise sensible selections?

This validation-only benchmark answers that question directly.

For random, physically valid channel sets at matched realised FLOP budgets it records:

- the sum of measured single-channel harms;
- the actual joint raw-pruning harm;
- the interaction residual;
- and, for a frozen subset of sets, the result after one minimal recovery epoch.

The discovery half calibrates an additive predictor; the confirmation half evaluates it.
No test data are used. This benchmark is more informative than comparing only five
hand-selected methods.


In [ ]:
from pathlib import Path
import os, sys, json, subprocess, platform, hashlib
import numpy as np
import pandas as pd
import torch

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

candidates = [
    os.environ.get("SABER_REPO"),
    "/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression",
    str(Path.cwd()),
]
REPO = None
for candidate in candidates:
    if not candidate:
        continue
    p = Path(candidate).expanduser()
    if (p / "src/saber").is_dir() and (p / "config").is_dir():
        REPO = p.resolve()
        break
if REPO is None:
    raise FileNotFoundError("Set SABER_REPO to the saber-ids-method repository.")
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Repository:", REPO)
print("Device:", DEVICE)
print("Python:", sys.version.split()[0], "|", platform.platform())


In [ ]:
from copy import deepcopy
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error
from scipy.stats import spearmanr
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, Subset

from src.saber.bridge_ciciot import load_bridge
from src.saber.deep_model import DeepCNN1D
from src.saber.taxonomy import ciciot2023_taxonomy, DEFAULT_COST_PROFILES
from src.saber.metrics import full_model_audit, action_weighted_boundary_inversion_rate
from src.saber.adapters import collect_logits
from src.saber.surgery import prune_cnn1d_channels, profile_forward_flops
from src.saber.postg5 import calibrate_prefix_count, additive_set_summary

OUT = REPO / "results/saber/22_set_interaction"
OUT.mkdir(parents=True, exist_ok=True)

SETS_PER_BUDGET = 40
BUDGETS_SHALLOW = [0.25, 0.40, 0.55]
BUDGETS_DEEP = [0.40]
RECOVERY_SETS_PER_ARCH = 12
BALANCED_PER_CLASS = 256
SEED = 22026

TRAIN_LOADER, VAL_LOADER, TEST_LOADER, SHALLOW, CLASS_NAMES = load_bridge()
taxonomy = ciciot2023_taxonomy(CLASS_NAMES)
graph = pd.read_csv(REPO / "results/saber/14_risk_graph/asvg_edges_robust.csv")
SHALLOW = SHALLOW.to(DEVICE).eval()

deep_path = REPO / "models/ciciot2023/deepcnn1d_g5_seed0.pt"
if not deep_path.exists():
    raise FileNotFoundError(deep_path)
DEEP = DeepCNN1D(len(CLASS_NAMES))
DEEP.load_state_dict(torch.load(deep_path, map_location="cpu", weights_only=False)["state_dict"])
DEEP = DEEP.to(DEVICE).eval()

shallow_single = pd.read_csv(
    REPO / "results/saber/16_score_validation/score_and_causal_harm.csv"
).merge(
    pd.read_csv(REPO / "results/saber/16b_score_v2/v2_scores.csv")[
        ["group_id", "v_c"]
    ],
    on="group_id", how="left",
)
deep_single = pd.read_csv(REPO / "results/saber/20_depth_probe/deep_group_scores.csv").merge(
    pd.read_csv(REPO / "results/saber/20_depth_probe/deep_group_harm.csv"),
    on=["group_id", "module_path", "channel_index"], how="inner",
)


In [ ]:
def balanced_validation_loader(per_class=BALANCED_PER_CLASS):
    X, y = VAL_LOADER.dataset.tensors
    rng = np.random.default_rng(SEED)
    idx = []
    y_np = y.numpy()
    for c in range(len(CLASS_NAMES)):
        candidates = np.flatnonzero(y_np == c)
        take = min(per_class, len(candidates))
        idx.extend(rng.choice(candidates, size=take, replace=False).tolist())
    idx = np.asarray(sorted(idx))
    return DataLoader(
        TensorDataset(X[idx], y[idx]),
        batch_size=1024,
        shuffle=False,
    )

AUDIT_LOADER = balanced_validation_loader()
EXAMPLE = next(iter(AUDIT_LOADER))[0][:8].to(DEVICE)

@torch.no_grad()
def audit_model(model, teacher_logits, labels):
    logits, observed, _ = collect_logits(model, AUDIT_LOADER, device=DEVICE)
    if not np.array_equal(labels, observed):
        raise RuntimeError("Validation audit labels changed.")
    audit = full_model_audit(logits, labels, taxonomy, DEFAULT_COST_PROFILES)
    awbir, _ = action_weighted_boundary_inversion_rate(
        teacher_logits, logits, labels, graph
    )
    audit["awbir"] = float(awbir)
    return logits, audit

T_SHALLOW, Y_AUDIT, _ = collect_logits(SHALLOW, AUDIT_LOADER, device=DEVICE)
T_DEEP, Y_DEEP, _ = collect_logits(DEEP, AUDIT_LOADER, device=DEVICE)
if not np.array_equal(Y_AUDIT, Y_DEEP):
    raise RuntimeError("Architecture audit labels differ.")

TEACHER_AUDIT = {
    "shallow": full_model_audit(T_SHALLOW, Y_AUDIT, taxonomy, DEFAULT_COST_PROFILES),
    "deep": full_model_audit(T_DEEP, Y_AUDIT, taxonomy, DEFAULT_COST_PROFILES),
}


In [ ]:
def valid_random_order(groups, rng, minimum_width):
    shuffled = groups.iloc[rng.permutation(len(groups))]
    remaining = groups.groupby("module_path")["group_id"].count().to_dict()
    order = []
    for row in shuffled.itertuples():
        layer = str(row.module_path)
        if remaining[layer] - 1 < minimum_width:
            continue
        remaining[layer] -= 1
        order.append((layer, int(row.channel_index), str(row.group_id)))
    return order

def prune_from_order(teacher, order, k, minimum_width):
    prune_map = {}
    for layer, channel, _ in order[:int(k)]:
        prune_map.setdefault(layer, []).append(channel)
    prune_map = {layer: sorted(channels) for layer, channels in prune_map.items()}
    student, _ = prune_cnn1d_channels(
        teacher, prune_map, EXAMPLE, minimum_remaining_per_layer=minimum_width
    )
    return student.to(DEVICE)

def realised_fraction(model, dense_flops):
    value = profile_forward_flops(model, EXAMPLE)["flops_per_item"]
    return 1.0 - float(value) / float(dense_flops)

def one_epoch_recovery(model, subset_loader, class_weights):
    torch.manual_seed(SEED)
    model = model.to(DEVICE).train()
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss(weight=class_weights)
    for x, y in subset_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        loss = loss_fn(model(x), y)
        loss.backward()
        opt.step()
    return model.eval()

# Fixed 10% recovery subset and class weights.
train_y = TRAIN_LOADER.dataset.tensors[1].numpy()
counts = np.bincount(train_y, minlength=len(CLASS_NAMES))
w = np.zeros_like(counts, dtype=float)
w[counts > 0] = 1.0 / np.sqrt(counts[counts > 0])
w[counts > 0] /= w[counts > 0].mean()
CLASS_WEIGHTS = torch.tensor(w, dtype=torch.float32, device=DEVICE)
g = torch.Generator().manual_seed(SEED)
recover_idx = torch.randperm(len(TRAIN_LOADER.dataset), generator=g)[:len(TRAIN_LOADER.dataset)//10]
RECOVERY_LOADER = DataLoader(
    Subset(TRAIN_LOADER.dataset, recover_idx.tolist()),
    batch_size=1024,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
)


In [ ]:
RESULT_PATH = OUT / "set_level_causal_benchmark.csv"
MEMBERSHIP_PATH = OUT / "set_membership.csv"
result_rows = pd.read_csv(RESULT_PATH).to_dict("records") if RESULT_PATH.exists() else []
membership_rows = pd.read_csv(MEMBERSHIP_PATH).to_dict("records") if MEMBERSHIP_PATH.exists() else []
done = {(r["architecture"], float(r["budget"]), int(r["set_seed"])) for r in result_rows}

architectures = [
    {
        "name": "shallow",
        "teacher": SHALLOW,
        "single": shallow_single,
        "budgets": BUDGETS_SHALLOW,
        "minimum_width": 4,
        "teacher_logits": T_SHALLOW,
    },
    {
        "name": "deep",
        "teacher": DEEP,
        "single": deep_single,
        "budgets": BUDGETS_DEEP,
        "minimum_width": 8,
        "teacher_logits": T_DEEP,
    },
]

for spec in architectures:
    dense_flops = profile_forward_flops(spec["teacher"], EXAMPLE)["flops_per_item"]
    for budget in spec["budgets"]:
        for local_seed in range(SETS_PER_BUDGET):
            set_seed = SEED + local_seed + int(1000 * budget) + (10000 if spec["name"] == "deep" else 0)
            key = (spec["name"], float(budget), int(set_seed))
            if key in done:
                continue
            rng = np.random.default_rng(set_seed)
            order = valid_random_order(spec["single"], rng, spec["minimum_width"])
            calibration = calibrate_prefix_count(
                order,
                target_reduction=budget,
                build_model=lambda k, order=order, spec=spec: prune_from_order(
                    spec["teacher"], order, k, spec["minimum_width"]
                ),
                realised_reduction=lambda model, dense_flops=dense_flops: realised_fraction(
                    model, dense_flops
                ),
                tolerance=0.015,
            )
            selected = order[:calibration.prefix_length]
            gids = [gid for _, _, gid in selected]
            student = prune_from_order(
                spec["teacher"], order, calibration.prefix_length, spec["minimum_width"]
            )
            _, raw = audit_model(student, spec["teacher_logits"], Y_AUDIT)
            additive = additive_set_summary(gids, spec["single"])
            teacher_audit = TEACHER_AUDIT[spec["name"]]
            split = "discovery" if local_seed % 2 == 0 else "confirmation"
            set_id = f"{spec['name']}_b{int(100*budget)}_s{set_seed}"
            row = {
                "set_id": set_id,
                "architecture": spec["name"],
                "budget": budget,
                "set_seed": set_seed,
                "split": split,
                "prefix_length": calibration.prefix_length,
                "realised_flops": calibration.realised_reduction,
                **additive,
                "raw_awbir": raw["awbir"],
                "raw_hsr_delta": raw["hsr_balanced_soc"] - teacher_audit["hsr_balanced_soc"],
                "raw_fine_f1_loss": teacher_audit["fine_macro_f1"] - raw["fine_macro_f1"],
                "raw_family_f1_loss": teacher_audit["family_macro_f1"] - raw["family_macro_f1"],
                "raw_attack_to_benign": raw["attack_to_benign_rate"],
                "raw_benign_to_attack": raw["benign_to_attack_rate"],
            }
            result_rows.append(row)
            membership_rows.extend(
                {
                    "set_id": set_id,
                    "architecture": spec["name"],
                    "budget": budget,
                    "set_seed": set_seed,
                    "module_path": layer,
                    "channel_index": channel,
                    "group_id": gid,
                }
                for layer, channel, gid in selected
            )
            pd.DataFrame(result_rows).to_csv(RESULT_PATH, index=False)
            pd.DataFrame(membership_rows).to_csv(MEMBERSHIP_PATH, index=False)
            print(set_id, "realised=", round(calibration.realised_reduction, 4),
                  "AWBIR=", round(raw["awbir"], 4))

sets = pd.DataFrame(result_rows)
display(sets.groupby(["architecture", "budget", "split"]).size())


In [ ]:
# Discovery-calibrated additive model, evaluated only on confirmation sets.
from sklearn.metrics import r2_score, mean_absolute_error

MAPPINGS = [
    ("sum_harm_awbir", "raw_awbir"),
    ("sum_harm_hsr_balanced_soc", "raw_hsr_delta"),
    ("sum_harm_fine_macro_f1", "raw_fine_f1_loss"),
    ("sum_harm_family_macro_f1", "raw_family_f1_loss"),
]
report = []
prediction_rows = []
for (architecture, budget), frame in sets.groupby(["architecture", "budget"]):
    discovery = frame[frame["split"] == "discovery"]
    confirmation = frame[frame["split"] == "confirmation"]
    for predictor, target in MAPPINGS:
        model = LinearRegression().fit(discovery[[predictor]], discovery[target])
        pred = model.predict(confirmation[[predictor]])
        report.append({
            "architecture": architecture,
            "budget": budget,
            "predictor": predictor,
            "target": target,
            "n_discovery": len(discovery),
            "n_confirmation": len(confirmation),
            "slope": float(model.coef_[0]),
            "intercept": float(model.intercept_),
            "confirmation_spearman": float(spearmanr(confirmation[target], pred).correlation),
            "confirmation_r2": float(r2_score(confirmation[target], pred)),
            "confirmation_mae": float(mean_absolute_error(confirmation[target], pred)),
        })
        for set_id, observed, estimate in zip(confirmation["set_id"], confirmation[target], pred):
            prediction_rows.append({
                "set_id": set_id,
                "architecture": architecture,
                "budget": budget,
                "target": target,
                "observed": float(observed),
                "additive_prediction": float(estimate),
                "interaction_residual": float(observed - estimate),
            })

report = pd.DataFrame(report)
predictions = pd.DataFrame(prediction_rows)
report.to_csv(OUT / "additive_confirmation_report.csv", index=False)
predictions.to_csv(OUT / "set_interaction_residuals.csv", index=False)
display(report)


In [ ]:
# Frozen minimal-recovery subset: first N confirmation sets at 40% per architecture.
RECOVERY_PATH = OUT / "minimal_recovery_reordering.csv"
recovery_rows = pd.read_csv(RECOVERY_PATH).to_dict("records") if RECOVERY_PATH.exists() else []
recovery_done = {r["set_id"] for r in recovery_rows}
membership = pd.DataFrame(membership_rows)

for spec in architectures:
    candidates = sets[
        (sets["architecture"] == spec["name"])
        & np.isclose(sets["budget"], 0.40)
        & (sets["split"] == "confirmation")
    ].sort_values("set_seed").head(RECOVERY_SETS_PER_ARCH)
    for row in candidates.itertuples():
        if row.set_id in recovery_done:
            continue
        members = membership[membership["set_id"] == row.set_id]
        prune_map = {
            layer: sorted(frame["channel_index"].astype(int).tolist())
            for layer, frame in members.groupby("module_path")
        }
        student, _ = prune_cnn1d_channels(
            spec["teacher"], prune_map, EXAMPLE,
            minimum_remaining_per_layer=spec["minimum_width"],
        )
        student = one_epoch_recovery(student, RECOVERY_LOADER, CLASS_WEIGHTS)
        _, recovered = audit_model(student, spec["teacher_logits"], Y_AUDIT)
        recovery_rows.append({
            "set_id": row.set_id,
            "architecture": spec["name"],
            "budget": row.budget,
            "raw_awbir": row.raw_awbir,
            "recovered_awbir": recovered["awbir"],
            "raw_family_macro_f1": TEACHER_AUDIT[spec["name"]]["family_macro_f1"] - row.raw_family_f1_loss,
            "recovered_family_macro_f1": recovered["family_macro_f1"],
            "raw_fine_macro_f1": TEACHER_AUDIT[spec["name"]]["fine_macro_f1"] - row.raw_fine_f1_loss,
            "recovered_fine_macro_f1": recovered["fine_macro_f1"],
            "recovered_attack_to_benign": recovered["attack_to_benign_rate"],
            "recovered_benign_to_attack": recovered["benign_to_attack_rate"],
        })
        pd.DataFrame(recovery_rows).to_csv(RECOVERY_PATH, index=False)
        print("recovered", row.set_id)

recovery = pd.DataFrame(recovery_rows)
reorder = []
for architecture, frame in recovery.groupby("architecture"):
    reorder.append({
        "architecture": architecture,
        "n_sets": len(frame),
        "raw_vs_recovered_awbir_spearman": float(
            spearmanr(frame["raw_awbir"], frame["recovered_awbir"]).correlation
        ),
        "raw_vs_recovered_family_f1_spearman": float(
            spearmanr(frame["raw_family_macro_f1"], frame["recovered_family_macro_f1"]).correlation
        ),
        "awbir_spread_raw": float(frame["raw_awbir"].max() - frame["raw_awbir"].min()),
        "awbir_spread_recovered": float(
            frame["recovered_awbir"].max() - frame["recovered_awbir"].min()
        ),
    })
reorder = pd.DataFrame(reorder)
reorder.to_csv(OUT / "recovery_reordering_summary.csv", index=False)
display(reorder)


In [ ]:
# Mechanism verdict, not a success/failure gate for a pruning method.
confirm = report[report["target"].isin(["raw_awbir", "raw_family_f1_loss"])]
raw_additive_supported = bool(
    (confirm["confirmation_spearman"] >= 0.60).mean() >= 0.5
)
recovery_reorders = bool(
    (reorder["raw_vs_recovered_awbir_spearman"] < 0.50).any()
    or (reorder["awbir_spread_recovered"] < 0.5 * reorder["awbir_spread_raw"]).any()
)
verdict = {
    "benchmark": "set_level_interaction_and_recovery_reordering",
    "raw_additivity_supported": raw_additive_supported,
    "recovery_reorders_or_equalizes": recovery_reorders,
    "interpretation": (
        "If raw additivity is supported but recovery reorders/equalizes, the "
        "channel-to-deployment dissociation is recovery-mediated. If raw additivity "
        "also fails, set interactions contribute independently."
    ),
}
(OUT / "mechanism_verdict.json").write_text(json.dumps(verdict, indent=2), encoding="utf-8")
print(json.dumps(verdict, indent=2))
